In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [2]:
train = pd.read_csv('battery_V1.csv')
test = pd.read_csv('test_battery_V1.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


In [3]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 495.244
[200]	valid_0's l2: 473.161
[300]	valid_0's l2: 459.136
[400]	valid_0's l2: 449.201
[500]	valid_0's l2: 441.182
[600]	valid_0's l2: 434.419
[700]	valid_0's l2: 428.042
[800]	valid_0's l2: 423.287
[900]	valid_0's l2: 418.425
[1000]	valid_0's l2: 415.092
Did not meet early stopping. Best iteration is:
[999]	valid_0's l2: 415.09
── Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 505.63
[200]	valid_0's l2: 486.885
[300]	valid_0's l2: 475.211
[400]	valid_0's l2: 465.44
[500]	valid_0's l2: 458.005
[600]	valid_0's l2: 452.242
[700]	valid_0's l2: 446.959
[800]	valid_0's l2: 442.546
[900]	valid_0's l2: 438.434
[1000]	valid_0's l2: 434.991
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 434.991
── Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 467.038
[200]	valid_0's l2: 446.225
[300]	vali

In [5]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 9.2484


In [6]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V2.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
